In [0]:
%run ../common/environmental_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
df_results = (
    spark.read.table(bronze_table)
    .drop('url')
    .withColumnRenamed('constructorId', 'constructor_id')
    .withColumnRenamed('driverId', 'driver_id')
    .withColumnRenamed('raceName', 'race_name')
    .withColumnRenamed('positionText', 'finish_position_text')
    .withColumnRenamed('date', 'race_date')
    .withColumnRenamed('grid', 'grid_position')
    .withColumnRenamed('laps', 'completed_laps')
    .withColumnRenamed('number', 'car_number')
    .withColumnRenamed('position', 'finish_position')
    .filter(
        (F.col('season').isNotNull()) &
        (F.col('round').isNotNull()) &
        (F.col('constructor_id').isNotNull()) &
        F.col('driver_id').isNotNull()
        )
    .dropDuplicates(['season' , 'round' , 'constructor_id' , 'driver_id'])
    .withColumn('race_name' , F.initcap(F.col('race_name')))
)
df_results.write.format('delta').mode('overwrite').saveAsTable(silver_table)